# Corrected Research Revision

**The stricter audit found 23 self-links across 16 of 192 historical study graphs.** Those files are preserved but quarantined. The corrected export retains 176 graphs. Old '192 verified' claims meant shallow artifact checks, not scientific validation.

The original analysis is archived below and disabled by default. Some old cells write output files or depend on unavailable experiments. Use the maintained revision section at the bottom. Run `python export_research_viewer.py` first.

Historical prompts changed participants' stated spoken language as well as instruction language. Fixing the calculations does not remove that confound. No maintained notebook cell makes an API call.


In [ ]:
from IPython.core.magic import register_cell_magic

# Opt in deliberately: the archived exploratory cells may write old outputs.
RUN_LEGACY_ANALYSIS = False

@register_cell_magic
def legacy_archive(line, cell):
    if RUN_LEGACY_ANALYSIS:
        result = get_ipython().run_cell(cell)
        result.raise_error()
    else:
        print('Skipped archival cell; use the corrected revision section below.')


# Beginner Guide To This Notebook

This notebook is the exploratory analysis companion to the Python scripts in the repo.

How to read it:
- `generate_personas.py` creates or loads the people used in the study.
- `generate_networks.py` asks the model to form friendships and saves graphs.
- `analyze_networks.py` computes the metrics.
- this notebook is where those saved metrics are compared, plotted, and interpreted.

What to expect below:
- some cells load precomputed CSVs rather than regenerating graphs
- some sections compare real vs generated networks
- some sections focus on homophily by demographic variables
- later sections inspect differences across prompting methods or models

If you are new, do not try to understand every plotting cell first. Start by finding:
1. where data is loaded
2. which subset of metrics is selected
3. what each figure is trying to compare

This notebook is more presentation-oriented than the scripts. The scripts do the repeatable pipeline work; this notebook helps a human inspect and explain the results.


In [ ]:
%%legacy_archive
%load_ext autoreload
%autoreload 2

from constants_and_utils import *

from generate_personas import *
from generate_networks import *
from analyze_networks import *
from plotting import *
from network_datasets import *

import matplotlib.pyplot as plt

# Analyze networks

## How To Read The First Analysis Section

This block focuses on comparing generated networks to each other and to real networks.

What the next cells usually do:
- load one or more CSV files from `stats/`
- filter to a subset of methods or models
- rename metrics into cleaner labels
- generate comparison plots

Important note for this capstone repo:
- some of the early notebook cells come from older pre-capstone experiments
- those cells now skip missing legacy datasets instead of crashing
- the capstone-relevant sections for `RQ1-RQ4` are at the bottom of the notebook

When reading each code cell, ask three questions:
1. which datasets are being loaded?
2. which methods/models are being kept?
3. which metric is being plotted or summarized?



In [ ]:
%%legacy_archive
# Combine homophily outputs from multiple conditions into one dataframe so
# later cells can compare methods and models side by side.
# Load previously computed metric tables instead of regenerating networks.
# This keeps the notebook focused on analysis and comparison.
from pathlib import Path


def load_csv_if_present(path, label, missing_inputs):
    if Path(path).exists():
        return pd.read_csv(path)
    missing_inputs.append((label, path))
    return None


def filter_available_save_names(df, requested_names, label):
    if df is None or 'save_name' not in df.columns:
        print(f'{label}: dataframe is unavailable.')
        return None, []
    available = set(df['save_name'].dropna().unique())
    kept = [name for name in requested_names if name in available]
    missing = [name for name in requested_names if name not in available]
    if missing:
        print(f'{label}: skipping missing save_names -> {missing}')
    if not kept:
        print(f'{label}: no requested save_names are available.')
        return None, []
    return df[df.save_name.isin(kept)].copy(), kept


missing_inputs = []
network_df = pd.read_csv('stats/real/network_metrics.csv')
metric_dfs = [network_df]
homophily_dfs = []

# load network metrics and homophily for generated networks
model = 'gpt-3.5-turbo'
for method in ['global', 'local', 'sequential']:
    metric_df = load_csv_if_present(f'stats/{method}_{model}/network_metrics.csv', f'{method}_{model} metrics', missing_inputs)
    homophily_df_part = load_csv_if_present(f'stats/{method}_{model}/homophily.csv', f'{method}_{model} homophily', missing_inputs)
    if metric_df is not None:
        metric_dfs.append(metric_df)
    if homophily_df_part is not None:
        homophily_dfs.append(homophily_df_part)
for extension in ['_ALL_SHUFFLED', '_w_interests', '_only_interests', '_temp06', '_temp10', '_prompt_all', '_w_reason']:
    metric_df = load_csv_if_present(f'stats/sequential_{model}{extension}/network_metrics.csv', f'sequential_{model}{extension} metrics', missing_inputs)
    homophily_df_part = load_csv_if_present(f'stats/sequential_{model}{extension}/homophily.csv', f'sequential_{model}{extension} homophily', missing_inputs)
    if metric_df is not None:
        metric_dfs.append(metric_df)
    if homophily_df_part is not None:
        homophily_dfs.append(homophily_df_part)

model = 'gpt-4o'
for extension in ['', '_prompt_all', '_n5', '_n5_prompt_all']:
    metric_df = load_csv_if_present(f'stats/sequential_{model}{extension}/network_metrics.csv', f'sequential_{model}{extension} metrics', missing_inputs)
    homophily_df_part = load_csv_if_present(f'stats/sequential_{model}{extension}/homophily.csv', f'sequential_{model}{extension} homophily', missing_inputs)
    if metric_df is not None:
        metric_dfs.append(metric_df)
    if homophily_df_part is not None:
        homophily_dfs.append(homophily_df_part)

for model in ['llama3.1-8b', 'llama3.1-70b', 'gemma2-9b', 'gemma2-27b']:
    for extension in ['', '_n5']:
        metric_df = load_csv_if_present(f'stats/sequential_{model}{extension}/network_metrics.csv', f'sequential_{model}{extension} metrics', missing_inputs)
        homophily_df_part = load_csv_if_present(f'stats/sequential_{model}{extension}/homophily.csv', f'sequential_{model}{extension} homophily', missing_inputs)
        if metric_df is not None:
            metric_dfs.append(metric_df)
        if homophily_df_part is not None:
            homophily_dfs.append(homophily_df_part)

network_df = pd.concat(metric_dfs, ignore_index=True)
homophily_df = pd.concat(homophily_dfs, ignore_index=True) if homophily_dfs else pd.DataFrame()

if missing_inputs:
    print('Skipped missing legacy inputs:')
    for label, path in missing_inputs:
        print(f'- {label}: {path}')
print('Loaded network rows:', len(network_df))
print('Loaded homophily rows:', len(homophily_df))



In [ ]:
%%legacy_archive
# Rename metric keys into labels that are easier to read on tables and plots.
homophily_df = pd.concat(homophily_dfs)
print(homophily_df.groupby('save_name').size())

In [ ]:
%%legacy_archive
def rename_network_metrics(n):
    if n == 'prop_nodes_lcc':
        return 'Prop nodes LCC'
    n = n.split('_')
    n = ' '.join(n)
    return n.capitalize()

for n in network_df['metric_name'].unique():
    print(n, rename_network_metrics(n))
    
network_df['metric_name'] = network_df['metric_name'].apply(rename_network_metrics)
network_df.head()

In [ ]:
%%legacy_archive
# GPT 3.5
x_to_keep = ['Density', 'Avg clustering coef', 'Prop nodes LCC', 'Avg shortest path', 'Modularity']
network_df[network_df['metric_name'].isin(x_to_keep) & network_df.save_name.str.contains('gpt-3.5')].groupby(['save_name', 'metric_name'])['_metric_value'].agg(['count', 'mean', 'sem']).round(3)

In [ ]:
%%legacy_archive
# non-GPT 3.5 
network_df[network_df['metric_name'].isin(x_to_keep) & ~network_df.save_name.str.contains('gpt-3.5')].groupby(['save_name', 'metric_name'])['_metric_value'].agg(['count', 'mean', 'sem']).round(3)

In [ ]:
%%legacy_archive
# main results - GPT 3.5 metrics, separate plots
to_keep = ['real', 'global_gpt-3.5-turbo', 'local_gpt-3.5-turbo', 'sequential_gpt-3.5-turbo']
kept_df, kept_names = filter_available_save_names(network_df, to_keep, 'GPT 3.5 metrics comparison')
if kept_df is not None:
    plot_metrics_separately(kept_df, plot_type='default', x_to_keep=x_to_keep,
                            simplify_legend=True, legend_mapper=None)



In [ ]:
%%legacy_archive
# GPT 3.5 - main results on homophily
to_keep = ['global_gpt-3.5-turbo', 'local_gpt-3.5-turbo', 'sequential_gpt-3.5-turbo', 'sequential_gpt-3.5-turbo_ALL_SHUFFLED', 'sequential_gpt-3.5-turbo_w_interests']
kept_df, kept_names = filter_available_save_names(homophily_df, to_keep, 'GPT 3.5 homophily comparison')
if kept_df is not None:
    kept_df = kept_df[kept_df['demo'] != 'age']
    make_plot(kept_df, plot_homophily=True, plot_type='default', y_lim=(0.7, 2.2), figsize=(14, 5), dodge=0.65)



In [ ]:
%%legacy_archive
# compare GPT 3.5 vs GPT 4o
to_keep = ['real', 'sequential_gpt-3.5-turbo', 'sequential_gpt-4o', 'sequential_gpt-4o_n5']
kept_df, kept_names = filter_available_save_names(network_df, to_keep, 'GPT 3.5 vs GPT 4o metrics')
legend_mapper = {'real': 'Real',
                 'sequential_gpt-3.5-turbo': 'Seq. (GPT-3.5)',
                 'sequential_gpt-4o': 'Seq. (GPT-4o)',
                 'sequential_gpt-4o_n5': 'Seq.+$\lambda$ (GPT-4o)'}
pastel_palette = sns.color_palette("pastel")
colors = {'real': pastel_palette[0],
          'sequential_gpt-3.5-turbo': pastel_palette[3],
          'sequential_gpt-4o': pastel_palette[4],
          'sequential_gpt-4o_n5': pastel_palette[5]}
if kept_df is not None:
    make_plot(kept_df, plot_homophily=False, plot_type='default', legend_mapper=legend_mapper, palette=colors,
              x_to_keep=x_to_keep, figsize=(11, 4), legend_pos=(1,1))

homophily_kept_df, homophily_kept_names = filter_available_save_names(homophily_df, [name for name in to_keep if name != 'real'], 'GPT 3.5 vs GPT 4o homophily')
if homophily_kept_df is not None:
    homophily_kept_df = homophily_kept_df[homophily_kept_df['demo'] != 'age']
    make_plot(homophily_kept_df, plot_homophily=True, plot_type='default', legend_mapper=legend_mapper, palette=colors,
              dodge=0.55, figsize=(11, 5), legend_pos=(1,1))



In [ ]:
%%legacy_archive
# compare GPT 3.5 vs Llama
to_keep = ['real', 'sequential_gpt-3.5-turbo', 'sequential_llama3.1-8b', 'sequential_llama3.1-8b_n5', 'sequential_llama3.1-70b', 'sequential_llama3.1-70b_n5']
kept_df, kept_names = filter_available_save_names(network_df, to_keep, 'GPT 3.5 vs Llama metrics')
legend_mapper = {'real': 'Real',
                 'sequential_gpt-3.5-turbo': 'Seq. (GPT-3.5)',
                 'sequential_llama3.1-8b': 'Seq. (Llama3.1 8B)',
                 'sequential_llama3.1-8b_n5': 'Seq.+$\lambda$ (Llama3.1 8B)',
                 'sequential_llama3.1-70b': 'Seq. (Llama3.1 70B)',
                 'sequential_llama3.1-70b_n5': 'Seq.+$\lambda$ (Llama3.1 70B)'}
colors = {'real': pastel_palette[0],
          'sequential_gpt-3.5-turbo': pastel_palette[3],
          'sequential_llama3.1-8b': pastel_palette[4],
          'sequential_llama3.1-8b_n5': pastel_palette[5],
          'sequential_llama3.1-70b': pastel_palette[6],
          'sequential_llama3.1-70b_n5': pastel_palette[7]}
if kept_df is not None:
    make_plot(kept_df, plot_homophily=False, plot_type='default', legend_mapper=legend_mapper, palette=colors,
              x_to_keep=x_to_keep, figsize=(11, 4), legend_pos=(1,1), dodge=0.67)

homophily_kept_df, homophily_kept_names = filter_available_save_names(homophily_df, [name for name in to_keep if name != 'real'], 'GPT 3.5 vs Llama homophily')
if homophily_kept_df is not None:
    homophily_kept_df = homophily_kept_df[homophily_kept_df['demo'] != 'age']
    make_plot(homophily_kept_df, plot_homophily=True, plot_type='default', legend_mapper=legend_mapper, palette=colors,
              dodge=0.65, figsize=(11, 5), legend_pos=(1,1))



In [ ]:
%%legacy_archive
# compare GPT 3.5 vs Gemma
to_keep = ['real', 'sequential_gpt-3.5-turbo', 'sequential_gemma2-9b', 'sequential_gemma2-27b', 'sequential_gemma2-27b_n5']
kept_df, kept_names = filter_available_save_names(network_df, to_keep, 'GPT 3.5 vs Gemma metrics')
legend_mapper = {'real': 'Real',
                 'sequential_gpt-3.5-turbo': 'Seq. (GPT-3.5)',
                 'sequential_gemma2-9b': 'Seq. (Gemma2 9B)',
                 'sequential_gemma2-27b': 'Seq. (Gemma2 27B)',
                 'sequential_gemma2-27b_n5': 'Seq.+$\lambda$ (Gemma2 27B)'}
colors = {'real': pastel_palette[0],
          'sequential_gpt-3.5-turbo': pastel_palette[3],
          'sequential_gemma2-9b': pastel_palette[4],
          'sequential_gemma2-27b': pastel_palette[5],
          'sequential_gemma2-27b_n5': pastel_palette[6]}
if kept_df is not None:
    make_plot(kept_df, plot_homophily=False, plot_type='default', legend_mapper=legend_mapper, palette=colors,
              x_to_keep=x_to_keep, figsize=(11, 4), legend_pos=(1,1), dodge=0.65)

homophily_kept_df, homophily_kept_names = filter_available_save_names(homophily_df, [name for name in to_keep if name != 'real'], 'GPT 3.5 vs Gemma homophily')
if homophily_kept_df is not None:
    homophily_kept_df = homophily_kept_df[homophily_kept_df['demo'] != 'age']
    make_plot(homophily_kept_df, plot_homophily=True, plot_type='default', legend_mapper=legend_mapper, palette=colors,
              figsize=(11, 5), dodge=0.6, legend_pos=(1,1))



In [ ]:
%%legacy_archive
# homophily with interests
to_keep = ['sequential_gpt-3.5-turbo', 'sequential_gpt-3.5-turbo_w_interests', 'sequential_gpt-3.5-turbo_only_interests']
legend_mapper = {'sequential_gpt-3.5-turbo': 'Seq.',
                 'sequential_gpt-3.5-turbo_w_interests': 'Seq. w interests',
                 'sequential_gpt-3.5-turbo_only_interests': 'Seq. only interests'}
colors = {'sequential_gpt-3.5-turbo': pastel_palette[3],
          'sequential_gpt-3.5-turbo_w_interests': pastel_palette[4],
          'sequential_gpt-3.5-turbo_only_interests': pastel_palette[5]}
kept_df, kept_names = filter_available_save_names(homophily_df, to_keep, 'GPT 3.5 interests homophily')
if kept_df is not None:
    kept_df = kept_df[kept_df['demo'] != 'age']
    make_plot(kept_df, plot_homophily=True, plot_type='default', legend_mapper=legend_mapper, palette=colors,
              figsize=(11, 5), dodge=0.53)



In [ ]:
%%legacy_archive
# compare across temperatures
to_keep = ['real', 'sequential_gpt-3.5-turbo', 'sequential_gpt-3.5-turbo_temp06', 'sequential_gpt-3.5-turbo_temp10']
legend_mapper = {'real': 'Real',
                 'sequential_gpt-3.5-turbo': 'Seq. (temp=0.8)',
                 'sequential_gpt-3.5-turbo_temp06': 'Seq. (temp=0.6)',
                 'sequential_gpt-3.5-turbo_temp10': 'Seq. (temp=1.0)'}
colors = {'real': pastel_palette[0],
          'sequential_gpt-3.5-turbo': pastel_palette[3],
          'sequential_gpt-3.5-turbo_temp06': pastel_palette[4],
          'sequential_gpt-3.5-turbo_temp10': pastel_palette[5]}
kept_df, kept_names = filter_available_save_names(network_df, to_keep, 'Temperature metrics comparison')
if kept_df is not None:
    make_plot(kept_df, plot_homophily=False, plot_type='default', legend_mapper=legend_mapper, palette=colors,
              x_to_keep=x_to_keep, figsize=(11, 4), legend_pos=(1.252,1))

homophily_kept_df, homophily_kept_names = filter_available_save_names(homophily_df, [name for name in to_keep if name != 'real'], 'Temperature homophily comparison')
if homophily_kept_df is not None:
    homophily_kept_df = homophily_kept_df[homophily_kept_df['demo'] != 'age']
    make_plot(homophily_kept_df, plot_homophily=True, plot_type='default', legend_mapper=legend_mapper, palette=colors,
              figsize=(11, 5), legend_pos=(1,1), dodge=0.53)



In [ ]:
%%legacy_archive
# testing "Pay all to all demographics"
to_keep = ['real', 'sequential_gpt-3.5-turbo', 'sequential_gpt-3.5-turbo_prompt_all', 'sequential_gpt-4o', 'sequential_gpt-4o_prompt_all', 'sequential_gpt-3.5-turbo_w_reason']
kept_df, kept_names = filter_available_save_names(network_df, to_keep, 'Prompt-all metrics comparison')
legend_mapper = {'real': 'Real',
                 'sequential_gpt-3.5-turbo': 'Seq. (GPT-3.5)',
                 'sequential_gpt-3.5-turbo_prompt_all': 'Seq.+"all" (GPT-3.5)',
                 'sequential_gpt-4o': 'Seq. (GPT-4o)',
                 'sequential_gpt-4o_prompt_all': 'Seq.+"all" (GPT-4o)',
                 'sequential_gpt-3.5-turbo_w_reason': 'Seq.+"reason" (GPT-3.5)'}
colors = {'real': pastel_palette[0],
          'sequential_gpt-3.5-turbo': pastel_palette[3],
          'sequential_gpt-3.5-turbo_prompt_all': pastel_palette[4],
          'sequential_gpt-4o': pastel_palette[5],
          'sequential_gpt-4o_prompt_all': pastel_palette[6],
          'sequential_gpt-3.5-turbo_w_reason': pastel_palette[7]}
if kept_df is not None:
    make_plot(kept_df, plot_homophily=False, plot_type='default', legend_mapper=legend_mapper, palette=colors,
              x_to_keep=x_to_keep, figsize=(11, 4), dodge=0.65)

homophily_kept_df, homophily_kept_names = filter_available_save_names(homophily_df, [name for name in to_keep if name != 'real'], 'Prompt-all homophily comparison')
if homophily_kept_df is not None:
    homophily_kept_df = homophily_kept_df[homophily_kept_df['demo'] != 'age']
    make_plot(homophily_kept_df, plot_homophily=True, plot_type='default', figsize=(11, 5), legend_mapper=legend_mapper, palette=colors,
              dodge=0.62, legend_pos=(1,1))



In [ ]:
%%legacy_archive
to_keep = ['real', 'global_gpt-3.5-turbo', 'local_gpt-3.5-turbo', 'sequential_gpt-3.5-turbo']
kept_df, kept_names = filter_available_save_names(network_df, to_keep, 'Degree histogram comparison')

if kept_df is not None:
    palette = get_pallete(kept_df)
    fig, axes = plt.subplots(2, 2, figsize=(8, 6))
    fig.subplots_adjust(hspace=0.3)
    bins = np.arange(0, 1.01, 0.05)
    print(bins)

    for i, ax in enumerate(axes.flatten()):
        if i >= len(kept_names):
            ax.axis('off')
            continue
        name = kept_names[i]
        degrees = kept_df[(kept_df.save_name == name) & (kept_df.metric_name == 'Degree centrality')]['_metric_value'].values
        print(name, len(degrees), degrees.max())
        ax.hist(degrees, bins=bins, color=palette[name])
        ax.set_yscale('log')
        if i % 2 == 0:
            ax.set_ylabel('Num nodes (log)', fontsize=14)
        if i >= 2:
            ax.set_xlabel('Degree', fontsize=14)
        ax.set_title(get_short_name(name), fontsize=16)
        ax.grid(alpha=0.2)



In [ ]:
%%legacy_archive
real_df = load_real_homophily(same_group=True)
real_df.sort_values('save_name').round(2)

In [ ]:
%%legacy_archive
# age homophily
fn = os.path.join(PATH_TO_TEXT_FILES, 'us_50_gpt4o_w_interests.json')
with open(fn) as f:
    personas = json.load(f)

method = 'sequential'
model = 'gpt-3.5-turbo'
list_of_G = load_list_of_graphs(f'{method}_{model}', 0, 30, directed=False)
if len(list_of_G) == 0:
    print(f'Skipping age homophily; no graphs found for {method}_{model}.')
else:
    plot_expected_vs_observed_age_gaps(list_of_G, personas)



In [ ]:
%%legacy_archive
def report_isolation_index(method, model):
    list_of_G = load_list_of_graphs(f'{method}_{model}', 0, 30, directed=False)
    if len(list_of_G) == 0:
        print(f'{method}_{model}: no graphs found, skipping isolation report')
        return
    print(f'{method}_{model}: found {len(list_of_G)} graphs')
    isolation = []
    exposure_c = []
    exposure_l = []
    for G in list_of_G:
        i, c, l = compute_isolation_index(G, personas)
        isolation.append(i)
        exposure_c.append(c)
        exposure_l.append(l)
    print(f'Isolation: {np.mean(isolation):0.3f}, {np.std(isolation)/np.sqrt(len(isolation)):0.3f}')
    print(f'Avg exposure, conservative: {np.mean(exposure_c):0.3f}, {np.std(exposure_c)/np.sqrt(len(exposure_c)):0.3f}')
    print(f'Avg exposure, liberal: {np.mean(exposure_l):0.3f}, {np.std(exposure_l)/np.sqrt(len(exposure_l)):0.3f}')

model = 'gpt-3.5-turbo'
for method in ['global', 'local', 'sequential']:
    print(method)
    report_isolation_index(method, model)
    print()



In [ ]:
%%legacy_archive
def report_polarization(method, model):
    list_of_G = load_list_of_graphs(f'{method}_{model}', 0, 30, directed=False)
    if len(list_of_G) == 0:
        print(f'{method}_{model}: no graphs found, skipping polarization report')
        return
    print(f'{method}_{model}: found {len(list_of_G)} graphs')
    pol = []
    for G in list_of_G:
        p = compute_polarization(G, personas)
        pol.append(p)
    print(f'Polarization: mean={np.mean(pol):0.3f}, se={np.std(pol)/np.sqrt(len(pol)):0.3f}')

model = 'gpt-3.5-turbo'
for method in ['global', 'local', 'sequential']:
    print(method)
    report_polarization(method, model)
    print()



## How To Read The Classical-Model Comparison Section

This section switches from LLM-generated networks to classical graph models such as Erdos-Renyi, Barabasi-Albert, and Watts-Strogatz.

The goal here is not to generate new personas. Instead, it asks whether traditional graph generators can match the same structural statistics that the LLM-generated graphs produce.

In this section, pay attention to:
- which real-network baseline is being matched
- which graph-model hyperparameters are fitted
- how close each synthetic model gets to the real or LLM-generated metrics


# Comparison to classical models

In [ ]:
%%legacy_archive
metrics = ['Density', 'Avg clustering coef', 'Prop nodes LCC', 'Avg shortest path', 'Modularity', 'Degree centrality']
real_mean = network_df[network_df['save_name'] == 'real'].groupby('metric_name')['_metric_value'].agg(['mean', 'sem'])
real_mean

In [ ]:
%%legacy_archive
n = 50
total_edges = n * (n-1) / 2
real_density = real_mean.loc['Density']['mean'] # real networks' mean density
exp_edges = total_edges * real_density
print(exp_edges)

In [ ]:
%%legacy_archive
# erdos-renyi
p = real_density
edges = []
for s in range(30):
    G = nx.erdos_renyi_graph(n, p, seed=s)
    edges.append(len(G.edges()))
    fn = os.path.join(PATH_TO_TEXT_FILES, f'er_{s}.adj')
    nx.write_adjlist(G, fn)
plt.hist(edges, bins=20)
plt.show()

In [ ]:
%%legacy_archive
# barabasi-albert
n = 50
best_m = None 
best_diff = 1e6
for m in range(1, 8):
    G = nx.barabasi_albert_graph(n, m, seed=s)
    num_edges = len(G.edges())
    diff = np.abs(len(G.edges()) - exp_edges)
    if diff < best_diff:
        best_diff = diff 
        best_m = m 
print('Using m =', best_m)

edges = []
for s in range(30):
    G = nx.barabasi_albert_graph(n, best_m, seed=s)
    edges.append(len(G.edges()))
    fn = os.path.join(PATH_TO_TEXT_FILES, f'ba_{s}.adj')
    nx.write_adjlist(G, fn)
print(set(edges))
plt.hist(edges, bins=20)
plt.show()

In [ ]:
%%legacy_archive
# watts-strogatz 
for k in np.arange(4, 16, 2):
    for p in np.arange(0.05, 0.16, 0.05):
        for s in range(5):
            G = nx.watts_strogatz_graph(n, k, p, seed=s)
            assert len(G.edges()) == (n*k)/2

In [ ]:
%%legacy_archive
# fit k (number of neighbors)
best_k = None 
best_diff = 1e6
p = 0.01
for k in np.arange(8, 13, 2):
    G = nx.watts_strogatz_graph(n, k, p, seed=s)
    num_edges = len(G.edges())
    assert num_edges == (n*k)/2
    diff = np.abs(num_edges - exp_edges)
    print(k, diff)
    if diff < best_diff:
        best_diff = diff 
        best_k = k 
print('Using k =', best_k)

In [ ]:
%%legacy_archive
# fit p (rewiring probability)
best_p = None 
best_diff = 1e6
p_options = np.arange(0.01, 0.5, 0.01)
p_clustering = []
for p in p_options:
    clustering = []
    for s in range(30):
        G = nx.watts_strogatz_graph(n, best_k, p)
        clustering.append(nx.average_clustering(G))
    p_clustering.append(np.mean(clustering))
plt.figure(figsize=(6,4))
plt.plot(p_options, p_clustering)
diff = np.abs(p_clustering - real_mean.loc['Avg clustering coef']['mean'])
plt.plot(p_options, diff)
min_val = p_options[np.argmin(diff)]
best_p = min_val.round(3)
print('best p = ', best_p)
ymin, ymax = plt.ylim()
plt.vlines([min_val], ymin, ymax, color='grey')
plt.show()

In [ ]:
%%legacy_archive
for s in range(30):
    G = nx.watts_strogatz_graph(n, best_k, best_p, seed=s)
    fn = os.path.join(PATH_TO_TEXT_FILES, f'ws_{s}.adj')
    nx.write_adjlist(G, fn)

In [ ]:
%%legacy_archive
# summarize network models
min_seed = 0
max_seed = 29
for model in ['er', 'ba', 'ws']: 
    list_of_G, t1, t2 = load_list_of_graphs(model, min_seed, max_seed+1, directed=False, include_ts=True)
    print(f'{model}: found {len(list_of_G)} graphs ({t1} TO {t2})')
    summarize_network_metrics(list_of_G, None, None, model, demos=False)
    print()

In [ ]:
%%legacy_archive
network_df = pd.read_csv('stats/real/network_metrics.csv')
metric_dfs = [network_df]

# load network metrics for generated networks when the files are present
model = 'gpt-3.5-turbo'
for method in ['global', 'local', 'sequential']:
    path = f'stats/{method}_{model}/network_metrics.csv'
    if os.path.exists(path):
        metric_dfs.append(pd.read_csv(path))
    else:
        print(f'Skipping missing metrics file: {path}')

for model in ['er', 'ba', 'ws']:
    path = f'stats/{model}/network_metrics.csv'
    if os.path.exists(path):
        metric_dfs.append(pd.read_csv(path))
    else:
        print(f'Skipping missing metrics file: {path}')

network_df = pd.concat(metric_dfs, ignore_index=True)
print(network_df.groupby('save_name').size())



In [ ]:
%%legacy_archive
results = []
available_save_names = set(network_df['save_name'].dropna().unique())
for method in ['er', 'ba', 'ws', 'global', 'local', 'sequential']:
    if len(method) > 2:
        save_name = f'{method}_gpt-3.5-turbo'
    else:
        save_name = method
    if save_name not in available_save_names:
        print(f'Skipping comparison for {save_name}; metrics are not available in this workspace snapshot.')
        continue
    for metric in ['density', 'avg_clustering_coef', 'prop_nodes_lcc', 'avg_shortest_path', 'modularity', 'degree_centrality']:
        mean_diff, mean_diff_norm, ks_stat, ks_pval = compare_network_metrics(network_df, metric, save_name)
        results.append({'method': method, 'metric': metric, 'mean_diff': mean_diff, 'mean_diff_norm': mean_diff_norm,
                        'ks_stat': ks_stat, 'ks_pval': ks_pval})
df = pd.DataFrame(results)
df.round(3)



## RQ1-RQ3: Culture, Demographics, and Model Consistency Across All Four Methods (UA Capstone)

This section should be read as one combined evidence block.

How to interpret it:
- Step 2 gave the sequential cultural-study baseline
- Step 3 added `global`, `local`, and `iterative` under the same culture/model/seed structure
- together, those two studies are what let the final project answer RQ1, RQ2, and RQ3 across all four methods rather than only the sequential method

What this section is for:
- read the final cultural-study summary for the sequential baseline
- read the method-study summary for the other three methods
- confirm that all generated artifacts passed verification
- inspect method-level density and homophily summaries



In [ ]:
%%legacy_archive
from IPython.display import Markdown, display
import pandas as pd

with open('stats/cultural_study/research_answers.md', 'r', encoding='utf-8') as f:
    display(Markdown(f.read()))

with open('stats/method_study/research_answers.md', 'r', encoding='utf-8') as f:
    display(Markdown(f.read()))

step2_condition_summary = pd.read_csv('stats/cultural_study/condition_summary.csv')
step2_verification = pd.read_csv('stats/cultural_study/verification_summary.csv') if __import__('os').path.exists('stats/cultural_study/verification_summary.csv') else None
step3_method_summary = pd.read_csv('stats/method_study/method_summary.csv')
step3_verification = pd.read_csv('stats/method_study/verification_summary.csv')

step2_top_same_ratio = step2_condition_summary[(step2_condition_summary['table'] == 'homophily') & (step2_condition_summary['metric_name'] == 'same_ratio')][['culture', 'model', 'demo', '_metric_value', 'std_value']].sort_values('_metric_value', ascending=False).head(12)
step3_density = step3_method_summary[(step3_method_summary['table'] == 'network') & (step3_method_summary['metric_name'] == 'density')][['method', '_metric_value', 'std_value']].sort_values('_metric_value', ascending=False)
step3_same_ratio = step3_method_summary[(step3_method_summary['table'] == 'homophily') & (step3_method_summary['metric_name'] == 'same_ratio')][['method', 'demo', '_metric_value', 'std_value']].sort_values(['method', '_metric_value'], ascending=[True, False])

if step2_verification is not None:
    print('Step 2 verification pass count:', int(step2_verification[['graph_exists', 'png_ok', 'homophily_ok', 'network_metrics_ok', 'node_count_ok', 'edge_count_ok']].all(axis=1).sum()), '/', len(step2_verification))
else:
    print('Step 2 verification summary file was not found in this workspace snapshot.')
print('Step 3 verification pass count:', int(step3_verification[['graph_exists', 'png_ok', 'homophily_ok', 'network_metrics_ok', 'node_count_ok', 'edge_count_ok']].all(axis=1).sum()), '/', len(step3_verification))
print('Methods covered by RQ1-RQ3 evidence: sequential, global, local, iterative')

display(step2_top_same_ratio)
display(step3_density)
display(step3_same_ratio.head(16))



## RQ4: Prompt-Language Effects With Culture Held Constant (UA Capstone)

This section isolates the fourth research question.

How to interpret it:
- culture is fixed to `us`
- personas stay the same
- the language of the prompt changes across English, Spanish, Hindi, and Japanese

What this section is for:
- read the final Step 4 language-study summary
- confirm the language-study outputs passed verification
- inspect how density and same-ratio values move across prompt languages



In [ ]:
%%legacy_archive
from IPython.display import Markdown, display
import pandas as pd

with open('stats/language_study/research_answers.md', 'r', encoding='utf-8') as f:
    display(Markdown(f.read()))

step4_language_summary = pd.read_csv('stats/language_study/language_summary.csv')
step4_verification = pd.read_csv('stats/language_study/verification_summary.csv')

step4_density = step4_language_summary[(step4_language_summary['table'] == 'network') & (step4_language_summary['metric_name'] == 'density')][['prompt_language', '_metric_value', 'std_value']].sort_values('_metric_value', ascending=False)
step4_same_ratio = step4_language_summary[(step4_language_summary['table'] == 'homophily') & (step4_language_summary['metric_name'] == 'same_ratio')][['prompt_language', 'demo', '_metric_value', 'std_value']].sort_values(['prompt_language', '_metric_value'], ascending=[True, False])

print('Step 4 verification pass count:', int(step4_verification[['graph_exists', 'png_ok', 'homophily_ok', 'network_metrics_ok', 'node_count_ok', 'edge_count_ok']].all(axis=1).sum()), '/', len(step4_verification))
display(step4_density)
display(step4_same_ratio.head(20))



## Corrected Inventory and Exclusions

A file hash is a fingerprint. This section checks the original graph bytes against the export manifest before presenting results. Quarantine means exclusion, not a silent repair or a new model run. Every retained graph has the same 50 IDs, nonzero edges, and no self-links.


In [ ]:
import hashlib
import json
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

revision = Path('stats/revision_v1')
manifest = json.loads((revision / 'manifest.json').read_text(encoding='utf-8'))
verification = pd.read_csv(revision / 'verification_summary.csv')
network = pd.read_csv(revision / 'network_metrics.csv')
homophily = pd.read_csv(revision / 'homophily.csv')
condition_summary = pd.read_csv(revision / 'condition_summary.csv')
model_pairs = pd.read_csv(revision / 'model_divergence.csv')
assert len(verification) == manifest['expected_runs'] == 192
assert len(network) == manifest['exported_runs'] == int(verification.passed.sum())
for filename, expected_hash in manifest['source_sha256'].items():
    assert hashlib.sha256(Path(filename).read_bytes()).hexdigest() == expected_hash, filename
print(f"Inventoried: {len(verification)}; retained: {len(network)}; quarantined: {manifest['quarantined']}; self-links: {manifest['self_links']}")
display(verification.groupby('study').agg(inventoried=('run_id', 'size'), retained=('passed', 'sum')))
display(verification.loc[~verification.passed, ['run_id', 'self_loops', 'loop_edges']])


## RQ1 and Method Comparisons

Hold model at GPT-4.1-mini and instructions at English. Each row keeps the method and country framing visible. Means and sample standard deviations describe available runs; they are not confidence intervals. At most two seeds and one US-structured roster cannot establish how real cultures behave. Different friendship budgets limit comparisons between methods.


In [ ]:
rq1 = condition_summary[(condition_summary.model == 'gpt-4.1-mini') & (condition_summary.study != 'language') & condition_summary.metric.isin(['density', 'avg_clustering_coef', 'prop_nodes_lcc'])]
display(rq1[['method', 'culture', 'metric', 'count', 'mean', 'std']])
display(Image(filename=str(revision / 'topology_comparison.png')))
display(Image(filename=str(revision / 'network_comparison.png')))


## RQ2: Mixing Is Not Causal Importance

Categorical Coleman scores adjust same-group incident ties by population share. Undefined scores remain missing. Age uses numeric assortativity, not a categorical Coleman ranking. The actual supplied attributes were gender, age, race/ethnicity, religion and political affiliation; names and interests were excluded by the study runners. Masking experiments and independent rosters remain necessary to test influence.


In [ ]:
rq2 = homophily[(homophily.model == 'gpt-4.1-mini') & (homophily.culture == 'us') & (homophily.study != 'language')]
display(rq2.groupby(['method', 'demographic', 'metric']).value.agg(['size', 'count', 'mean', 'std']))
print('Undefined homophily scores across retained runs:', int(homophily.value.isna().sum()))


## RQ3 and RQ4: Keep Scope and Confounds Visible

RQ3 compares undirected edges within a matched historical study, method, context, language and seed. Distance 0 means identical edges; 1 means every possible edge differs. Sparse unrelated graphs can have a small distance, so this is not a stand-alone realism score.

RQ4 holds country framing at US, but old prompts also assigned participant language. These results are descriptive, not an isolated instruction-language effect. Spanish is historical supplementary evidence; Portuguese is tested only in the new engineering pilot. Cross-provider evidence, bilingual review, source-grounded rosters and uncertainty analysis remain pending. Offline matched controls are available below, not a real-network benchmark.


In [ ]:
display(model_pairs.groupby(['study', 'method', 'culture', 'language', 'model_a', 'model_b']).edge_distance.agg(['count', 'mean', 'std']))
rq4 = condition_summary[(condition_summary.study == 'language') & (condition_summary.model == 'gpt-4.1-mini') & (condition_summary.metric == 'density')]
display(rq4[['method', 'culture', 'language', 'count', 'mean', 'std']])
print('No API calls. Corrected software does not imply confirmed research hypotheses.')


## Budgeted pilot and Astra review point
The original engineering pilot targets 20 full-50-person networks; a separately named extension adds four GPT-6 Sol US-English method checks, sharing the same $5 sub-cap inside $50 total. The draft main plan still targets 640 networks with Luna and Mini; Sol is not silently added to that matrix. These counts are planning targets, not conference acceptance requirements. Astra should review actual pilot evidence, prompts, costs and the proposed adult rosters before the main study. This notebook reads saved evidence and never dispatches API calls.

Pilot failures revealed reversed duplicate edges and incorrect response counts. Prompt/retry corrections were made during debugging. Source variants are shown rather than treated as frozen-protocol replications. Earlier responses lack parser annotations; missing logs are not zero failures. Scalar analysis now canonicalizes graph insertion order so seeded Louvain modularity survives serialization. Earlier measurements are preserved in insertion_order files; the table below uses recomputed verified metrics.


In [ ]:
# Inspect the versioned design without invoking any model.
from paid_study import build_cells
import pandas as pd
pilot_cells = build_cells('pilot') + build_cells('sol-pilot')
pilot_design = pd.DataFrame(pilot_cells)
main_design = pd.DataFrame(build_cells('main'))
display(pilot_design.groupby(['model', 'language']).size().rename('planned_networks'))
print(f'Pilot: {len(pilot_design)}; main draft: {len(main_design)}. Main collection awaits Astra review.')


In [ ]:
# Missing files mean no evidence exists yet; never manufacture a pass count.
from pathlib import Path
import sqlite3, json
from contextlib import closing
pilot_folder = Path('outputs/revision_budget_v1')
ledger = pilot_folder / 'budget.sqlite'
if ledger.exists():
    with closing(sqlite3.connect(ledger.resolve().as_uri() + '?mode=ro', uri=True)) as connection:
        display(pd.read_sql_query('SELECT phase, status, COUNT(*) AS attempts, SUM(cost) AS conservative_usd FROM attempts GROUP BY phase, status', connection))
else:
    print('Pilot not dispatched: no request ledger exists.')
from paid_study import verify_completed
finished = []
for cell in pilot_cells:
    result_path = pilot_folder / (cell['run_id'] + '.json')
    if not result_path.exists():
        continue
    record = verify_completed(result_path, cell)
    finished.append({**record['cell'], **record['verified_metrics'], 'prompt_variant': record['verified_source_variant'], 'status': record['status']})
print(f'Verified saved engineering networks: {len(finished)} / {len(pilot_design)}')
if finished:
    display(pd.DataFrame(finished)[['method','model','language','density','avg_clustering_coef','prop_nodes_lcc','prompt_variant','status']])
comparison_figure = revision / 'pilot_network_comparison.png'
if comparison_figure.exists():
    display(Image(filename=str(comparison_figure)))
    print('Measured US-English engineering graphs across available models; shared positions are algorithmic, not geographic or social distance.')


## Select Conditions and Reproduce the Viewer Plots
The local workbench supports multiple models, methods, country framings, instruction languages and seeds at once. Checkboxes take the intersection of these selections; unavailable combinations stay missing. Historical and engineering data are selectable separately. Every topology dot is a saved run. Heatmap values are descriptive means within a complete study/roster/prompt condition, with valid counts; age assortativity is not the same measure as categorical Coleman homophily. Downloads contain the selected rows and their source hashes.
The cell below is the notebook equivalent: edit condition_filter, then rerun. It reads the same exported measurements as the viewer and never generates a graph or spends tokens. It shows individual measurements without confidence intervals or significance claims. Sol currently covers only four US-English engineering methods; Luna also has Hindi, Japanese and Portuguese pilot runs.


In [ ]:
# Choose saved conditions here; this is analysis, not a paid experiment command.
import json, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
condition_filter = {'study': ['engineering_pilot'], 'model': ['gpt-6-sol', 'gpt-6-luna'],
                    'method': ['global', 'local', 'sequential', 'iterative'],
                    'culture': ['us'], 'language': ['english'], 'seed': [1000]}
viewer_data = json.loads(Path('viewer/public/data/networks.json').read_text(encoding='utf-8'))
chosen = [run for run in viewer_data['runs'] if all(run[key] in values for key, values in condition_filter.items())]
print('Saved matching runs:', len(chosen), '; missing combinations are not generated or filled with zeros.')
if chosen:
    selection_table = pd.DataFrame([{**{key: run[key] for key in condition_filter},
        'run_id': run['run_id'], 'prompt_variant': run.get('prompt_variant', 'legacy_pre_source_hash'),
        'source_sha256': run['source_sha256'], **run['metrics'], **run['homophily'],
        'age_numeric_assortativity': run['age_assortativity']} for run in chosen])
    display(selection_table)
    # Keep individual runs visible. Equal seed numbers do not imply equal prompt versions.
    plot_labels = [f"{run['model']} / {run['method']} / {run['culture']} / {run['language']} / {run['study']} / seed {run['seed']}" for run in chosen]
    fig, axes = plt.subplots(1, 2, figsize=(14, max(4, len(chosen) * .4)), layout='constrained')
    for axis, metric in zip(axes, ['density', 'avg_clustering_coef']):
        axis.scatter([run['metrics'][metric] for run in chosen], range(len(chosen)), color='#186e71')
        axis.set_yticks(range(len(chosen)), plot_labels)
        axis.set_xlim(-.02, 1.02)
        axis.set_title(metric)
        axis.grid(axis='x', alpha=.2)
    fig.suptitle('Selected saved runs / ' + ', '.join(condition_filter['study']) + '; descriptive, no confidence intervals')
    plt.show()


## Matched Offline Controls: What They Do and Do Not Prove
Three controls accompany each retained historical graph: uniform random edges with the same node/edge count, a declared equal-weight demographic-similarity heuristic, and degree-preserving edge swaps. These 528 synthetic controls cost no model tokens and are not extra LLM-generated networks. One control seed is exploratory, not an uncertainty analysis. The similarity heuristic includes the attributes being measured and resolves ties by ID; it is a reference rule, not a fitted or validated human friendship model. Rewiring flags report whether the requested swaps completed, not whether mixing converged.


In [ ]:
# Read controls only; generation and live API calls do not belong in analysis.
baseline_path = revision / 'baselines/matched_metrics.csv'
if baseline_path.exists():
    controls = pd.read_csv(baseline_path)
    display(controls.groupby('baseline').size().rename('control_graphs'))
    print('Synthetic control rows:', len(controls), '; not independent LLM runs.')
    display(controls.groupby('baseline')[['density','avg_clustering_coef','prop_nodes_lcc','modularity']].agg(['mean','min','max']))
else:
    print('Offline controls missing. Run make_matched_baselines.py before using this section.')


## Fresh 896-Run Study: Eight Repetitions and Calibration
The replacement study uses 50 fictional adults, four models and all four methods. It compares US, India, Japan and Brazil in English, then English, Hindi, Japanese and Brazilian Portuguese under US framing. The shared US-English baseline appears once: 4 models x 4 methods x 7 settings x 8 repetitions = 896. Candidate attributes stay in fixed English JSON while instruction language varies. Political orientation replaces US party labels, and race is excluded from this controlled roster. These are declared experimental choices, not national population estimates.
Fixture outputs are software checks, not model findings. Calibration targets 68 cells and stops at the separately approved $5 additional ceiling even if incomplete. Those runs are provisionally part of 896 only if prompts/protocol stay unchanged. The old $50 maximum is removed, but full collection needs a new finite ceiling and separate approval. Eight repetitions are not a power guarantee. Show individual pairs and count/mean/sample-standard-deviation/range, not independent-edge confidence intervals. See RESEARCH_896_HANDOFF.md and REVIEW_RESPONSE_MATRIX.md for limitations and bilingual-review requirements.
Independent AI-assisted review of all four instruction languages and actual retries is recorded in TRANSLATION_REVIEW.md. It clarified current common-friend counts and repaired Hindi/Japanese wording. The 80 base examples and 20 captured retry examples are offline review fixtures, not model evidence or human bilingual signoff.
Initial calibration exposed incomplete global repair, example-ID contamination, an empty response and a too-small per-person output cap. Those versions remain preserved and excluded from v5. The corrected parser preserves the full original valid tie set; only an initially empty answer gets a bounded nonresponse retry. V5 permits 512 completion tokens for per-person replies while still stopping on truncation. CALIBRATION_FINDINGS.md records the history and retained charges. The data-loading cell follows the runner's current retry_v5 paths.
Run inspect_calibration.py offline after collection for verified receipts, topology, controls, actual usage and an API-only runtime estimate. A readable PNG is not proof of scientific validity. Human bilingual signoff and main-study approval remain separate; do not interpret calibration completion as completion of 896 networks.
All 68 calibration graphs were collected. The logging/transport-recovery review preserves the original generation contract while recording actual execution hashes separately. One separately charged replacement reused 194 cached replies; both historical lost-response reservations remain fully counted. The tables below load verified current reports, not invented or historical replacement data.


In [ ]:
# This cell reads files only. It does not call OpenAI or generate model outputs.
from pathlib import Path
import json
import pandas as pd
import revision224 as fresh_study
fresh_folder = fresh_study.DESTINATION
fresh_results = fresh_study.ROOT / fresh_study.RESULTS
fresh_report_path = fresh_folder / 'report.json'
if fresh_report_path.exists():
    fresh_report = json.loads(fresh_report_path.read_text(encoding='utf-8'))
    current = fresh_report.get('protocol_sha256') == fresh_study.digest(fresh_study.load_config()) and fresh_report.get('source_sha256') == fresh_study.source_hashes()
    print('Offline review status:', fresh_report['status'], '| matches current sources:', current)
    print('Fresh run receipt files present (not verified by this count):', len(list(fresh_results.glob('*.json'))))
    if current and fresh_report['status'] == 'READY_FOR_ASTRA_CODE_REVIEW':
        fresh_checks = pd.read_csv(fresh_folder / 'verification_summary.csv')
        assert len(fresh_checks) == fresh_study.load_config()['main_networks_target'] and fresh_checks['passed'].all()
        display(fresh_checks.groupby(['model', 'method']).size().rename('offline_fixture_cells_checked').unstack())
        print('These are fixture checks, not research results. API calls for this preflight: 0.')
    else:
        print('Run revision224.py --preflight to replace stale or failed verification.')
else:
    print('Fresh offline preflight is missing. Run revision224.py --preflight.')
# Fresh analysis is deliberately separate from historical controls above.
fresh_analysis = fresh_study.ROOT / fresh_study.STATS
analysis_report_path = fresh_analysis / 'analysis_report.json'
if analysis_report_path.exists():
    analysis_report = json.loads(analysis_report_path.read_text(encoding='utf-8'))
    print('Fresh analysis:', analysis_report['status'], '| analyzed networks:', analysis_report.get('analyzed_networks', 0))
    receipts_current = all((fresh_results / name).exists() and fresh_study.hashlib.sha256((fresh_results / name).read_bytes()).hexdigest() == value for name, value in analysis_report.get('source_receipts_sha256', {}).items())
    analysis_current = analysis_report.get('protocol_sha256') == fresh_study.digest(fresh_study.load_config()) and analysis_report.get('analysis_source_sha256') == fresh_study.source_hashes()
    if analysis_report['status'] in {'PARTIAL', 'COMPLETE'} and analysis_current and receipts_current:
        display(pd.read_csv(fresh_analysis / 'network_metrics.csv'))
        display(pd.read_csv(fresh_analysis / 'paired_contrasts.csv'))
        display(pd.read_csv(fresh_analysis / 'contrast_summary.csv'))
        print('Contrasts are comparison minus reference, paired within a repetition. They are not significance tests.')
    else:
        print('No current verified V5 analysis. Run analyze_saved_study.py for ledger-free reanalysis; never fill absent results with old graphs.')
else:
    print('Run analyze_saved_study.py for ledger-free V5 verification and controls; no API calls.')
calibration_path = fresh_folder / 'calibration_report.json'
if calibration_path.exists():
    calibration = json.loads(calibration_path.read_text(encoding='utf-8'))
    current_calibration = calibration.get('protocol_sha256') == fresh_study.digest(fresh_study.load_config()) and calibration.get('source_sha256') == fresh_study.source_hashes()
    print('Calibration:', calibration['status'], '| matches current sources:', current_calibration, '| completed:', calibration.get('completed_calibration_networks', 0), '/ 68')
    print('Shared-ledger unresolved requests (including superseded protocols):', calibration.get('shared_ledger_unresolved_attempts', 'not reported'))
    print('Explicitly abandoned requests / retained reservation USD:', calibration.get('shared_ledger_abandoned_attempts', 'not reported'), calibration.get('shared_ledger_abandoned_reservation_usd', 'not reported'))
    if current_calibration and calibration['status'] == 'COMPLETE':
        print('Assumption-based whole-study forecast USD:', calibration['full_study_forecast_usd'])
        print('Estimated remaining generation USD:', calibration['forecast_remaining_usd'])
        print('Main-study approval is separate. Calibration does not establish statistical power or translation equivalence.')
inspection_path = fresh_analysis / 'calibration_inspection.json'
if inspection_path.exists():
    inspection = json.loads(inspection_path.read_text(encoding='utf-8'))
    inspection_current = inspection.get('protocol_sha256') == fresh_study.digest(fresh_study.load_config()) and inspection.get('generation_source_sha256') == fresh_study.generation_source_hashes(fresh_study.load_config()) and inspection.get('execution_source_sha256') == fresh_study.source_hashes()
    inspection_receipts_current = all((fresh_results / name).exists() and fresh_study.hashlib.sha256((fresh_results / name).read_bytes()).hexdigest() == value for name, value in inspection.get('receipt_sha256', {}).items())
    if inspection_current and inspection_receipts_current and inspection.get('status') in {'PARTIAL', 'COMPLETE'}:
        print('Calibration graphs verified:', inspection['verified_networks'])
        display(pd.DataFrame(inspection['networks'])[['model', 'method', 'culture', 'language', 'repetition', 'nodes', 'edges', 'parse_failures', 'conservative_charge_usd', 'request_roundtrip_seconds']])
        print('Serial API-only runtime forecast:', inspection['runtime_forecast'])
        print('A null forecast means calibration is incomplete. Completed forecasts and assumptions are documented in CALIBRATION_RESULTS.md.')
        print('Excludes local processing, rendering and analysis; contingency is a planning allowance, not a confidence interval.')
    else:
        print('Inspection incomplete or stale. Run inspect_calibration.py after bounded collection finishes.')


## October 2 design review: public offline reanalysis
The earlier 896-network proposal is superseded, not completed. V5 remains exploratory evidence. See plan.md for the corrected candidate prompts, staged allocation and pending approvals. This section reads only stats/public_v5_reanalysis; it never mixes revised fixtures with model outputs or opens the private billing ledger.
Density is constrained by nomination quotas in local/sequential/iterative, not exactly fixed. Local LCC is saturated in this calibration. Global degree-one share diagnoses matching-like behavior without imposing a preferred network. Portuguese first-response failures are a real compliance signal, not proof that the proposed singular wording fixes model behavior.
Reference draws are synthetic graphs conditioned on edge counts or degrees. Their percentiles are not treatment confidence intervals; completed rewiring is not proof of mixing. Power sensitivity assumes normal paired differences and does not establish that eight or 32 repetitions are adequate. Human bilingual review, paid probes, inferential validation and a new freeze remain pending.


In [ ]:
# Verify source data and table hashes before displaying the independent reanalysis.
from pathlib import Path
import json
import pandas as pd
from export_calibration_viewer import load_calibration, sha256
public_folder = Path('stats/public_v5_reanalysis')
public_report_path = public_folder / 'report.json'
if not public_report_path.exists():
    print('Run python -B analyze_saved_study.py first. No API key or private ledger is needed.')
else:
    public_report = json.loads(public_report_path.read_text(encoding='utf-8'))
    assert public_report['status'] == 'VERIFIED_V5_EXPLORATORY_REANALYSIS'
    _, _, checked_v5 = load_calibration()
    assert public_report['source_receipts_sha256'] == {path.name: sha256(path) for path, *_ in checked_v5}
    assert all(sha256(Path(name)) == digest for name, digest in public_report['analysis_source_sha256'].items())
    assert all(sha256(public_folder / name) == digest for name, digest in public_report['tables_sha256'].items())
    public_metrics = pd.read_csv(public_folder / 'network_metrics.csv')
    print('V5 observed graphs:', len(public_metrics), '| new model calls:', public_report['paid_calls'])
    display(public_metrics.groupby('method')[['density', 'prop_nodes_lcc', 'degree_one_share', 'modularity']].agg(['count', 'min', 'max']))
    compliance = pd.read_csv(public_folder / 'decision_compliance.csv')
    local_us = compliance[(compliance.model == 'gpt-6-luna') & (compliance.method == 'local') & (compliance.culture == 'us')]
    display(local_us.groupby('language').agg(decisions=('first_attempt_failed', 'size'), first_attempt_failures=('first_attempt_failed', 'sum')))
    print('Failures are clustered within graphs; decisions are not independent treatment repetitions.')
    reference_summary = pd.read_csv(public_folder / 'reference_summary.csv')
    display(reference_summary[reference_summary.metric == 'avg_clustering_coef'].head(12))
    display(pd.read_csv(public_folder / 'power_sensitivity.csv').pivot(index='pairs', columns='hypotheses', values='mdes_sd_of_paired_differences'))
    print('Holm family proposal and meaningful-effect thresholds require approval. These sensitivity bounds are not measured power.')


## Revised 104-graph calibration: separate evidence, not main results
This section reads only the revised calibration under contract `5fb3715550db`. The owner approved 104 graphs with a $10 additional ceiling; that does not authorize the main study. All graphs use the same 50 fictional adults. Four country framings are tested in English; Hindi, Japanese and Brazilian Portuguese are tested with US framing fixed. This is seven settings, not a full country-language factorial.
An empty graph is a recorded model response, not a missing file. Undefined homophily or modularity stays undefined. First-response failures and corrections are reported separately; decisions within a graph are not independent repetitions. The smaller model/configuration calibration counts do not establish equivalence or statistical power.
These cells never call an API. Run `python -B inspect_calibration_v6.py` after collection is stopped to verify receipts and refresh the report. The simulation intentionally remains unchanged. Human bilingual validation remains absent, as accepted by the owner.


In [ ]:
# Check the evidence version before treating a saved summary as current.
from pathlib import Path
import hashlib, json
import pandas as pd
import calibration_v6 as revised_calibration
v6_folder = Path('outputs/calibration_v6/5fb3715550db')
v6_report_file = v6_folder / 'inspection.json'
v6_report = None
if not v6_report_file.exists():
    print('No revised inspection yet. Do not substitute V5 or offline fixture graphs.')
else:
    v6_candidate = json.loads(v6_report_file.read_text(encoding='utf-8'))
    assert v6_candidate['contract_sha256'] == revised_calibration.frozen.digest(revised_calibration.contract())
    assert v6_candidate['inspector_sha256'] == hashlib.sha256(Path('inspect_calibration_v6.py').read_bytes()).hexdigest()
    assert {p.name for p in (v6_folder / 'runs').glob('*.json')} == set(v6_candidate['receipt_sha256']), 'Inspection is stale; refresh it after collection stops.'
    assert all(hashlib.sha256((v6_folder / 'runs' / name).read_bytes()).hexdigest() == digest for name, digest in v6_candidate['receipt_sha256'].items())
    assert all(hashlib.sha256((v6_folder / name).read_bytes()).hexdigest() == digest for name, digest in v6_candidate['tables_sha256'].items())
    v6_report = v6_candidate  # Publish to later cells only after every check passes.
    print(v6_report['status'], '| graphs:', v6_report['verified_networks'], '/', v6_report['expected_networks'])
    v6_networks = pd.DataFrame(v6_report['networks'])
    if not v6_networks.empty:
        display(v6_networks.groupby(['model', 'method']).agg(graphs=('run_id', 'size'), empty_graphs=('edges', lambda edges: int((edges == 0).sum())), median_density=('density', 'median'), cost_usd=('conservative_usd', 'sum')))
        display(v6_networks.groupby(['model', 'culture', 'language']).size().rename('graphs').reset_index())


In [ ]:
# Match country as well as language; do not pool different framings into RQ4.
if v6_report is not None:
    v6_compliance = pd.DataFrame(v6_report['decision_compliance'])
    if not v6_compliance.empty:
        display(v6_compliance[v6_compliance.first_failures > 0])
    print('Initial decisions / failures:', v6_report['first_decisions'], '/', v6_report['first_failures'])
    print('Empty graphs:', len(v6_report['empty_networks']))
    print('Undefined metric counts:', v6_report['undefined_metrics'])
    print('Completed-graph conservative usage USD:', v6_report['completed_network_conservative_usd'])
    print('Summed API hours:', v6_report['summed_api_hours'])
    print('Observed span, including pauses:', v6_report['observed_span_hours_including_pauses'])
    display(pd.DataFrame(v6_report['hypothetical_main_forecasts']))
    print('Forecasts assume unchanged prompts and fresh samples. They are not budgets, approvals, invoices or guarantees.')
    print('Non-US country forecasts for other models transfer Luna ratios; those cells were not measured directly.')
